[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/msfasha/307401-Big-Data/blob/main/20261/Module%202%20-%20Introduction%20to%20Hadoop/Practical/Beyond_Excel_10_Million_Rows.ipynb)

<div style="font-family:Arial,Helvetica,sans-serif;font-size:13px;color:#56616D;margin:8px 0 10px;">Big Data <span style="color:#56616D;">›</span> Module 2 · Introduction to Hadoop <span style="color:#56616D;">›</span> Practical lab</div>

<div style="font-family:Arial,Helvetica,sans-serif;border-top:3px solid #A83296;background:#F5EDF4;border-radius:0 0 8px 8px;padding:1.5rem 1.75rem;margin:0 0 20px;"><h1 style="margin:0 0 10px;padding:0;border:0;background:none;color:#493347;font-size:34px;line-height:1.2;">Beyond Excel:<br>Big Data on One PC</h1><div style="font-size:18px;color:#655263;line-height:1.45;">One laptop, 10 million rows, four tools — and how to tell when you really need a cluster</div></div>

**Case:** an online retailer with **10 million sales transactions**  
**Question we answer every way:** *What is the total revenue per region?*

<div style="font-family:Arial,Helvetica,sans-serif;background:#EFF4F7;border-left:3px solid #6C91A8;border-radius:0 6px 6px 0;padding:.7rem 1rem;margin:14px 0;color:#2F343B;font-size:15px;line-height:1.55;"><b>The big idea:</b> “Big data” does not automatically mean “cluster”. A single computer can answer many big-data questions — if we use all its cores and never load more than we need.</div>

<h3 style="font-family:Arial,Helvetica,sans-serif;margin:1.6rem 0 10px;padding:.5rem 1rem;background:#F5EDF4;border:0;border-left:3px solid #C8ABC3;border-radius:0 6px 6px 0;color:#2F343B;font-size:20px;line-height:1.3;font-weight:700;">Roadmap</h3>

| Step | Idea | Tools |
|---|---|---|
| 1 | **Parallelism** — many cores beat one | `multiprocessing` |
| 2 | **Break Excel's row limit** | Pandas |
| 3 | **Data bigger than RAM** — process in chunks | Pandas chunks |
| 4 | **Let a modern engine do it** | DuckDB · Polars · Dask |
| 5 | **Choose the simplest tool that works** | decision table → Hadoop / Spark |

<h2 style="font-family:Arial,Helvetica,sans-serif;margin:2.3rem 0 14px;padding:.75rem 1.1rem;background:#E9DDE7;border:0;border-left:3px solid #C8ABC3;border-radius:0 6px 6px 0;color:#493347;font-size:26px;line-height:1.25;font-weight:700;">0 · Setup</h2>

- Run the cells **top to bottom**.
- `N_ROWS = 10_000_000` creates a CSV of a few hundred MB (needs ~1 GB free disk).
- Slow or low-memory computer? Set `N_ROWS = 1_100_000` — still above Excel's limit, same concepts.
- Required: `numpy`, `pandas`, `matplotlib`. Optional (Section 5): `duckdb`, `polars`, `dask`.
- No cloud account, Hadoop or Spark needed.

In [ ]:
# %pip install -q duckdb polars "dask[dataframe]"    # uncomment if a package is missing, then restart the kernel
from pathlib import Path
import os, sys, time, json, subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

N_ROWS = 10_000_000            # use 1_100_000 on a slow computer
WRITE_BATCH = 250_000          # rows generated per batch
READ_CHUNK = 200_000           # rows per Pandas chunk (Section 4)
SEED = 2026
EXCEL_MAX_ROWS = 1_048_576

DATA_DIR = Path.cwd() / "local_bigdata_lab"
DATA_DIR.mkdir(exist_ok=True)
CSV_PATH = DATA_DIR / f"sales_{N_ROWS:,}.csv"
REGIONS = np.array(["North", "South", "East", "West"])
CORES = os.cpu_count()

print(f"CPU cores available to Python: {CORES}")
print(f"Rows planned: {N_ROWS:,}  ({N_ROWS / EXCEL_MAX_ROWS:.1f}x Excel's worksheet limit)")

<h2 style="font-family:Arial,Helvetica,sans-serif;margin:2.3rem 0 14px;padding:.75rem 1.1rem;background:#E9DDE7;border:0;border-left:3px solid #C8ABC3;border-radius:0 6px 6px 0;color:#493347;font-size:26px;line-height:1.25;font-weight:700;">1 · The big idea: big data on one machine</h2>

- **Volume** (a lot of data) and **velocity** (answers in milliseconds) are different problems.
- A cluster is *one possible* answer, not the definition of big data.
- Start from the **workload**, not from a technology name:
  1. What question must be answered?
  2. How much data must be in memory *at the same time*?
  3. How fast must the answer arrive?

![Start from the workload, not from a buzzword.](https://raw.githubusercontent.com/msfasha/307401-Big-Data/main/20261/Module%202%20-%20Introduction%20to%20Hadoop/Practical/images/decision.png)

<h2 style="font-family:Arial,Helvetica,sans-serif;margin:2.3rem 0 14px;padding:.75rem 1.1rem;background:#E9DDE7;border:0;border-left:3px solid #C8ABC3;border-radius:0 6px 6px 0;color:#493347;font-size:26px;line-height:1.25;font-weight:700;">2 · Experiment: many cores beat one</h2>

**Task:** count the multiples of 7 among 100 million numbers.  
**Two runs:** one process vs. one process per CPU core. Same answer, different time.

![Parallelism cuts elapsed time, but scheduling and communication add overhead.](https://raw.githubusercontent.com/msfasha/307401-Big-Data/main/20261/Module%202%20-%20Introduction%20to%20Hadoop/Practical/images/parallel.png)

In [ ]:
%%writefile local_bigdata_lab/parallel_demo.py
import os, time
from multiprocessing import Pool

def count_multiples_of_7(bounds):
    start, stop = bounds
    return sum(x % 7 == 0 for x in range(start, stop))

if __name__ == "__main__":
    LIMIT, cores = 100_000_000, os.cpu_count()
    step = LIMIT // cores
    pieces = [(i * step, LIMIT if i == cores - 1 else (i + 1) * step) for i in range(cores)]   # split

    t0 = time.perf_counter()
    answer_one = count_multiples_of_7((0, LIMIT))                  # 1 process does everything
    t_one = time.perf_counter() - t0

    t0 = time.perf_counter()
    with Pool(cores) as pool:
        answer_many = sum(pool.map(count_multiples_of_7, pieces))  # work in parallel, then combine
    t_many = time.perf_counter() - t0

    assert answer_one == answer_many
    print(f"1 process : {t_one:.2f} s")
    print(f"{cores} processes: {t_many:.2f} s   ->   {t_one / t_many:.1f}x faster")

In [ ]:
!python local_bigdata_lab/parallel_demo.py

<h3 style="font-family:Arial,Helvetica,sans-serif;margin:1.6rem 0 10px;padding:.5rem 1rem;background:#F5EDF4;border:0;border-left:3px solid #C8ABC3;border-radius:0 6px 6px 0;color:#2F343B;font-size:20px;line-height:1.3;font-weight:700;">What the experiment shows</h3>

- Separate **processes** run on separate cores → real speedup (at best ≈ number of cores).
- Python *threads* would not help here: they share one lock (the *GIL*).
- Speedup is **below** the core count: starting workers and combining answers costs time.
- A tiny job can even run *slower* in parallel — overhead beats the work.
- Winning pattern: **split → work in parallel → combine**.
- The engines in Section 5 do all of this for us, in fast native code.

<h3 style="font-family:Arial,Helvetica,sans-serif;margin:1.6rem 0 10px;padding:.5rem 1rem;background:#F5EDF4;border:0;border-left:3px solid #C8ABC3;border-radius:0 6px 6px 0;color:#2F343B;font-size:20px;line-height:1.3;font-weight:700;">The pattern has a name: Map and Reduce</h3>

- **Map:** compute a *partial* result for each piece of data.
- **Reduce:** merge the partial results into the final answer.
- It can run sequentially, on many cores, or on many machines — the idea is the same.
- *Hadoop MapReduce* is one distributed implementation; the pattern itself needs no cluster.

![MapReduce is a decompose-and-combine pattern, not a synonym for all parallel processing.](https://raw.githubusercontent.com/msfasha/307401-Big-Data/main/20261/Module%202%20-%20Introduction%20to%20Hadoop/Practical/images/mapreduce.png)

In [ ]:
from collections import Counter

partitions = [[("North", 100), ("South", 250)],      # piece 1 of the data
              [("North", 50),  ("West", 80)]]        # piece 2 of the data

def map_partition(rows):                  # MAP: partial totals for one piece
    totals = Counter()
    for region, revenue in rows:
        totals[region] += revenue
    return totals

partials = [map_partition(p) for p in partitions]    # could run on separate cores or machines
final = sum(partials, Counter())                     # REDUCE: merge the partial totals
print("Partial results:", [dict(p) for p in partials])
print("Final answer:   ", dict(final))

<h2 style="font-family:Arial,Helvetica,sans-serif;margin:2.3rem 0 14px;padding:.75rem 1.1rem;background:#E9DDE7;border:0;border-left:3px solid #C8ABC3;border-radius:0 6px 6px 0;color:#493347;font-size:26px;line-height:1.25;font-weight:700;">3 · Breaking Excel's row limit with Pandas</h2>

<div style="font-family:Arial,Helvetica,sans-serif;background:#EFF4F7;border-left:3px solid #6C91A8;border-radius:0 6px 6px 0;padding:.7rem 1rem;margin:14px 0;color:#2F343B;font-size:15px;line-height:1.55;"><b>Rule of thumb:</b> if the data fits in RAM, start with Pandas.</div>

<h3 style="font-family:Arial,Helvetica,sans-serif;margin:1.6rem 0 10px;padding:.5rem 1rem;background:#F5EDF4;border:0;border-left:3px solid #C8ABC3;border-radius:0 6px 6px 0;color:#2F343B;font-size:20px;line-height:1.3;font-weight:700;">Excel's ceiling</h3>

- A worksheet holds at most **1,048,576 rows**.
- Our file has **10,000,000** rows — about **9.5×** too many to open.
- Excel's Power Query / Data Model can go further, but a worksheet cannot show it.
- We do not need 10 million visible rows — we need **one aggregate**.

<h3 style="font-family:Arial,Helvetica,sans-serif;margin:1.6rem 0 10px;padding:.5rem 1rem;background:#F5EDF4;border:0;border-left:3px solid #C8ABC3;border-radius:0 6px 6px 0;color:#2F343B;font-size:20px;line-height:1.3;font-weight:700;">Create the dataset</h3>

- Columns: `transaction_id`, `region`, `product`, `units`, `unit_price_cents`, `revenue_cents`.
- Values are invented. Money is stored as **integer cents** to avoid rounding surprises.
- Rows are written **batch by batch**, so the full dataset is never held in memory.

In [ ]:
def generate_sales_csv(path, n_rows, batch_size=250_000, seed=2026):
    rng = np.random.default_rng(seed)
    products = np.array(["Laptop", "Phone", "Tablet", "Accessory"])
    prices = np.array([89900, 49900, 29900, 2500], dtype=np.int32)    # cents
    start = time.perf_counter()
    with path.open("w", encoding="utf-8", newline="") as f:
        for offset in range(0, n_rows, batch_size):
            size = min(batch_size, n_rows - offset)
            product_id = rng.integers(0, len(products), size=size)
            units = rng.integers(1, 6, size=size, dtype=np.int16)
            batch = pd.DataFrame({
                "transaction_id": np.arange(offset + 1, offset + size + 1),
                "region": REGIONS[rng.integers(0, len(REGIONS), size=size)],
                "product": products[product_id],
                "units": units,
                "unit_price_cents": prices[product_id],
                "revenue_cents": units.astype(np.int64) * prices[product_id],
            })
            batch.to_csv(f, index=False, header=(offset == 0))
    return time.perf_counter() - start

if not CSV_PATH.exists():
    print(f"Generated {N_ROWS:,} rows in {generate_sales_csv(CSV_PATH, N_ROWS, WRITE_BATCH, SEED):.1f} s")
else:
    print("Existing file found — delete it to regenerate:", CSV_PATH.name)
print(f"File size: {CSV_PATH.stat().st_size / 1024**2:,.0f} MiB")
print(pd.read_csv(CSV_PATH, nrows=5).to_string(index=False))

<h3 style="font-family:Arial,Helvetica,sans-serif;margin:1.6rem 0 10px;padding:.5rem 1rem;background:#F5EDF4;border:0;border-left:3px solid #C8ABC3;border-radius:0 6px 6px 0;color:#2F343B;font-size:20px;line-height:1.3;font-weight:700;">Pandas answers the question</h3>

- Read **only the two columns we need** (`usecols`) and store `region` as a compact `category`.
- Group by region and sum — three lines of code.

In [ ]:
def show_dollars(series):
    print((series / 100).map("${:,.2f}".format).to_string())

t0 = time.perf_counter()
sales = pd.read_csv(CSV_PATH, usecols=["region", "revenue_cents"],
                    dtype={"region": "category", "revenue_cents": "int64"})
reference = sales.groupby("region", observed=True)["revenue_cents"].sum().sort_index()
reference.index = reference.index.astype(object)       # plain labels, so other engines compare cleanly
benchmarks = {"Pandas (in memory)": time.perf_counter() - t0}

print(f"Rows loaded: {len(sales):,}   |   RAM used: {sales.memory_usage(deep=True).sum() / 1024**2:,.0f} MiB   |   "
      f"Time: {benchmarks['Pandas (in memory)']:.1f} s\n")
show_dollars(reference)
print(f"\nTotal revenue: ${reference.sum() / 100:,.2f}")
del sales          # free the memory before the next section

| | Excel worksheet | Pandas |
|---|---|---|
| Row limit | 1,048,576 | Limited only by RAM |
| Repeatable | Manual clicks | Re-run the same code on new data |
| Our 10 M rows | Cannot open | Answer in seconds |

- **Reflection:** the file is a few hundred MB — why did the DataFrame use far less?

<h2 style="font-family:Arial,Helvetica,sans-serif;margin:2.3rem 0 14px;padding:.75rem 1.1rem;background:#E9DDE7;border:0;border-left:3px solid #C8ABC3;border-radius:0 6px 6px 0;color:#493347;font-size:26px;line-height:1.25;font-weight:700;">4 · When data is bigger than RAM: chunking</h2>

<div style="font-family:Arial,Helvetica,sans-serif;background:#EFF4F7;border-left:3px solid #6C91A8;border-radius:0 6px 6px 0;padding:.7rem 1rem;margin:14px 0;color:#2F343B;font-size:15px;line-height:1.55;"><b>Idea:</b> read a piece, summarise it, throw the piece away. Peak memory = one chunk.</div>

- A **chunk** is a manageable group of rows.
- Keep only a small running summary (e.g. a total per region).
- Chunks are processed **one after another** — memory-efficient, but **not parallel**.

![Chunking limits peak memory; it does not by itself use multiple cores.](https://raw.githubusercontent.com/msfasha/307401-Big-Data/main/20261/Module%202%20-%20Introduction%20to%20Hadoop/Practical/images/chunks.png)

In [ ]:
def pandas_chunked_revenue(path, chunk_size=200_000):
    totals = pd.Series(dtype="int64")
    for chunk in pd.read_csv(path, chunksize=chunk_size,
                             usecols=["region", "revenue_cents"],
                             dtype={"region": "category", "revenue_cents": "int64"}):
        subtotal = chunk.groupby("region", observed=True)["revenue_cents"].sum()
        totals = totals.add(subtotal, fill_value=0)          # combine partial answers
    totals = totals.fillna(0).astype("int64").sort_index()
    totals.index = totals.index.astype(object)
    return totals

t0 = time.perf_counter()
chunked = pandas_chunked_revenue(CSV_PATH, READ_CHUNK)
benchmarks["Pandas chunks"] = time.perf_counter() - t0
pd.testing.assert_series_equal(chunked, reference, check_names=False, check_index_type=False)
print(f"Chunked Pandas: {benchmarks['Pandas chunks']:.1f} s — same answer as in-memory Pandas")

<h3 style="font-family:Arial,Helvetica,sans-serif;margin:1.6rem 0 10px;padding:.5rem 1rem;background:#F5EDF4;border:0;border-left:3px solid #C8ABC3;border-radius:0 6px 6px 0;color:#2F343B;font-size:20px;line-height:1.3;font-weight:700;">How the loop walks the file</h3>

```text
CSV on disk → [rows 1–5] → [rows 6–10] → [rows 11–15] → … → end of file
                loop 1        loop 2         loop 3
```
- `chunksize=` returns an **iterator**, not a DataFrame.
- Each loop reads the *next* chunk automatically; earlier chunks are not kept.

In [ ]:
from itertools import islice

reader = pd.read_csv(CSV_PATH, chunksize=5, usecols=["transaction_id", "region", "revenue_cents"])
for number, chunk in enumerate(islice(reader, 3), start=1):          # peek at 3 tiny chunks only
    print(f"Chunk {number}: transaction IDs {chunk.transaction_id.iloc[0]} to {chunk.transaction_id.iloc[-1]}")
reader.close()

<h3 style="font-family:Arial,Helvetica,sans-serif;margin:1.6rem 0 10px;padding:.5rem 1rem;background:#F5EDF4;border:0;border-left:3px solid #C8ABC3;border-radius:0 6px 6px 0;color:#2F343B;font-size:20px;line-height:1.3;font-weight:700;">Rule: you can chunk what you can combine</h3>

| Question | With chunks? | Why |
|---|---|---|
| Total revenue | ✅ Easy | Add the chunk subtotals |
| Average order value | ✅ Easy | Keep **sum and count**, divide at the end |
| Max revenue per region | ✅ Easy | Keep per-region maxima |
| Exact median | ⚠️ Hard | Median of chunk medians is **wrong** |
| Sort all 10 M rows | ⚠️ Hard | Sorted chunks joined ≠ sorted whole |

In [ ]:
chunk_a, chunk_b = [8, 2, 12], [10, 1, 5]
print("Sort each chunk, then join:", sorted(chunk_a) + sorted(chunk_b), "  ← NOT sorted")
print("Correct global sort:       ", sorted(chunk_a + chunk_b))
print("Sum of chunk sums:", sum(chunk_a) + sum(chunk_b), " = sum of everything:", sum(chunk_a + chunk_b))

<h3 style="font-family:Arial,Helvetica,sans-serif;margin:1.6rem 0 10px;padding:.5rem 1rem;background:#F5EDF4;border:0;border-left:3px solid #C8ABC3;border-radius:0 6px 6px 0;color:#2F343B;font-size:20px;line-height:1.3;font-weight:700;">Cut the input first</h3>

- Read only the **columns** you need (`usecols`).
- Choose compact **dtypes** (`category`, integers) — never at the cost of overflow or precision.
- For repeated analysis, store data as **Parquet**: columnar, compressed, reads only needed columns.

<h2 style="font-family:Arial,Helvetica,sans-serif;margin:2.3rem 0 14px;padding:.75rem 1.1rem;background:#E9DDE7;border:0;border-left:3px solid #C8ABC3;border-radius:0 6px 6px 0;color:#493347;font-size:26px;line-height:1.25;font-weight:700;">5 · Modern engines do the hard work for us</h2>

<div style="font-family:Arial,Helvetica,sans-serif;background:#EFF4F7;border-left:3px solid #6C91A8;border-radius:0 6px 6px 0;padding:.7rem 1rem;margin:14px 0;color:#2F343B;font-size:15px;line-height:1.55;"><b>Idea:</b> the same question, the same answer — with less code. Each engine splits the work and uses all cores automatically (Section 2's pattern, built in).</div>

| Tool | In one line | Reach for it when |
|---|---|---|
| **DuckDB** | SQL engine that runs inside Python | You think in SQL, or query files directly |
| **Polars** | Fast, lazy, multi-threaded DataFrames | You build DataFrame pipelines |
| **Dask** | Pandas split into partitions + scheduler | You want Pandas-style code that can scale out |

<h3 style="font-family:Arial,Helvetica,sans-serif;margin:1.6rem 0 10px;padding:.5rem 1rem;background:#F5EDF4;border:0;border-left:3px solid #C8ABC3;border-radius:0 6px 6px 0;color:#2F343B;font-size:20px;line-height:1.3;font-weight:700;">DuckDB — SQL straight on the file</h3>

- An **in-process analytical database**: no server, no install of a database system.
- Queries the CSV (or Parquet) **in place** — no import step.
- Optimises the query, runs on **all cores**, can spill to disk.
- Great for aggregations, joins, and **top-K** without a full sort.

In [ ]:
try:
    import duckdb
except ImportError:
    print("DuckDB not installed:  %pip install duckdb")
else:
    t0 = time.perf_counter()
    duck_result = duckdb.connect().execute("""
        SELECT region, SUM(revenue_cents)::BIGINT AS revenue_cents
        FROM read_csv_auto(?, header=true)
        GROUP BY region ORDER BY region
    """, parameters=[str(CSV_PATH)]).df().set_index("region")["revenue_cents"]
    benchmarks["DuckDB"] = time.perf_counter() - t0
    pd.testing.assert_series_equal(duck_result.sort_index(), reference, check_names=False, check_index_type=False)
    print(f"DuckDB: {benchmarks['DuckDB']:.1f} s — same answer")

    # Bonus: the 5 largest transactions — without sorting all 10 million rows into memory.
    print(duckdb.sql(f"""SELECT transaction_id, region, product, revenue_cents
                         FROM read_csv_auto('{CSV_PATH.as_posix()}')
                         ORDER BY revenue_cents DESC, transaction_id LIMIT 5""").df().to_string(index=False))

<h3 style="font-family:Arial,Helvetica,sans-serif;margin:1.6rem 0 10px;padding:.5rem 1rem;background:#F5EDF4;border:0;border-left:3px solid #C8ABC3;border-radius:0 6px 6px 0;color:#2F343B;font-size:20px;line-height:1.3;font-weight:700;">Polars — lazy and multi-threaded</h3>

- A DataFrame library written in **Rust**.
- **Lazy:** `scan_csv()` only *plans*; `.collect()` runs the optimised plan.
- Reads only the columns the query needs and runs on **all cores**.
- Great for column-heavy DataFrame pipelines.

In [ ]:
try:
    import polars as pl
except ImportError:
    print("Polars not installed:  %pip install polars")
else:
    t0 = time.perf_counter()
    polars_frame = (pl.scan_csv(str(CSV_PATH))            # plan only — nothing is read yet
                      .group_by("region")
                      .agg(pl.col("revenue_cents").sum())
                      .sort("region")
                      .collect())                          # execute the optimised plan
    benchmarks["Polars"] = time.perf_counter() - t0
    polars_result = pd.Series(dict(zip(polars_frame["region"].to_list(),
                                       polars_frame["revenue_cents"].to_list())), dtype="int64").sort_index()
    pd.testing.assert_series_equal(polars_result, reference, check_names=False, check_index_type=False)
    print(f"Polars: {benchmarks['Polars']:.1f} s — same answer")

<h3 style="font-family:Arial,Helvetica,sans-serif;margin:1.6rem 0 10px;padding:.5rem 1rem;background:#F5EDF4;border:0;border-left:3px solid #C8ABC3;border-radius:0 6px 6px 0;color:#2F343B;font-size:20px;line-height:1.3;font-weight:700;">Dask — Pandas in partitions</h3>

- Splits a DataFrame into **partitions** and schedules the work as tasks.
- The API looks like Pandas; nothing runs until `.compute()`.
- Runs on one PC today; the **same code** can run on a cluster later.
- Great for Pandas users who may outgrow one machine.

In [ ]:
try:
    import dask.dataframe as dd
except ImportError:
    print('Dask not installed:  %pip install "dask[dataframe]"')
else:
    t0 = time.perf_counter()
    ddf = dd.read_csv(str(CSV_PATH), blocksize="32MB", usecols=["region", "revenue_cents"])   # 32 MB partitions
    dask_result = ddf.groupby("region")["revenue_cents"].sum().compute(scheduler="threads")
    benchmarks["Dask"] = time.perf_counter() - t0
    dask_result = dask_result.sort_index().astype("int64")
    pd.testing.assert_series_equal(dask_result, reference, check_names=False, check_index_type=False)
    print(f"Dask: {benchmarks['Dask']:.1f} s — same answer")

<h3 style="font-family:Arial,Helvetica,sans-serif;margin:1.6rem 0 10px;padding:.5rem 1rem;background:#F5EDF4;border:0;border-left:3px solid #C8ABC3;border-radius:0 6px 6px 0;color:#2F343B;font-size:20px;line-height:1.3;font-weight:700;">Side by side</h3>

- Every result above was checked to be **identical** — correctness first, speed second.
- Timings are **single runs on your computer** (disk, cores, caching all matter).
- For a fair comparison: repeat, vary the order, and take the median.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.2))
labels = list(benchmarks)
seconds = [benchmarks[k] for k in labels]
ax.barh(labels, seconds, color="#356B8C")
ax.invert_yaxis()
ax.set_xlabel("Elapsed seconds (lower is better)")
ax.set_title("Revenue per region — measured on this computer")
for i, v in enumerate(seconds):
    ax.text(v, i, f" {v:.1f}s", va="center")
ax.set_xlim(0, max(seconds) * 1.2)
plt.tight_layout(); plt.show()

<h2 style="font-family:Arial,Helvetica,sans-serif;margin:2.3rem 0 14px;padding:.75rem 1.1rem;background:#E9DDE7;border:0;border-left:3px solid #C8ABC3;border-radius:0 6px 6px 0;color:#493347;font-size:26px;line-height:1.25;font-weight:700;">6 · Choosing the right tool</h2>

| Situation | First choice | Why |
|---|---|---|
| Fits in RAM | **Pandas** | Familiar, flexible |
| Bigger than RAM, simple sums/counts | **Pandas chunks** | Simple, predictable memory |
| Large CSV/Parquet + SQL | **DuckDB** | Query optimiser, all cores |
| DataFrame pipelines, many columns | **Polars** | Lazy plan, multi-threaded |
| Pandas-style code, may scale out later | **Dask** | Partitions + scheduler |
| Custom CPU-heavy Python | **Multiprocessing** | Bypasses the GIL |
| Beyond one machine, or needs fault tolerance | **Spark / Hadoop** | Scale out |


<h3 style="font-family:Arial,Helvetica,sans-serif;margin:1.6rem 0 10px;padding:.5rem 1rem;background:#F5EDF4;border:0;border-left:3px solid #C8ABC3;border-radius:0 6px 6px 0;color:#2F343B;font-size:20px;line-height:1.3;font-weight:700;">Six guardrails</h3>

1. **Chunking ≠ parallelism** — one chunk at a time uses one core.
2. **Parallelism ≠ distributed** — one PC already has many cores.
3. **MapReduce is one pattern**, not the only parallel model.
4. **Volume ≠ velocity** — an overnight report is not a 100 ms dashboard.
5. **Speedup is not guaranteed** — overhead, disk and uneven work can dominate.
6. **There is no magic size** — “big” depends on the workload and the machine.


<h3 style="font-family:Arial,Helvetica,sans-serif;margin:1.6rem 0 10px;padding:.5rem 1rem;background:#F5EDF4;border:0;border-left:3px solid #C8ABC3;border-radius:0 6px 6px 0;color:#2F343B;font-size:20px;line-height:1.3;font-weight:700;">When one PC is not enough</h3>

- Data exceeds the disk or compute of one machine.
- Results are needed faster than one machine can deliver.
- Many users, or the job must survive hardware failure.
- Data arrives as a continuous stream.
- → **Hadoop** (HDFS storage, YARN resources, MapReduce) and **Spark** (distributed engine) — next lecture.


<div style="font-family:Arial,Helvetica,sans-serif;background:#EFF4F7;border-left:3px solid #6C91A8;border-radius:0 6px 6px 0;padding:.7rem 1rem;margin:14px 0;color:#2F343B;font-size:15px;line-height:1.55;"><b>Before reaching for a cluster, ask:</b> What must be computed? How much must sit in memory at once? What delay is acceptable? Can streaming, partitioning, one fast engine or more cores solve it?</div>

<h2 style="font-family:Arial,Helvetica,sans-serif;margin:2.3rem 0 14px;padding:.75rem 1.1rem;background:#E9DDE7;border:0;border-left:3px solid #C8ABC3;border-radius:0 6px 6px 0;color:#493347;font-size:26px;line-height:1.25;font-weight:700;">7 · Practice</h2>

- **A. Bottleneck.** A 10 GB file, 8 GB RAM: when is chunked aggregation enough? Name two cases needing a different method.
- **B. New question.** Change the chunked function to compute revenue by **product**.
- **C. Chunk size.** Time `READ_CHUNK` = 50 000 / 200 000 / 500 000, three runs each. Why is the biggest not always fastest?
- **D. Parallelism.** Change `Pool(cores)` to `Pool(2)` in `parallel_demo.py`. When does adding workers stop helping?
- **E. Memory.** Remove `usecols` in Section 3. How much more RAM does Pandas need?
- **F. Recommend a tool** for: (a) nightly 10 GB sales report, (b) interactive analysis of Parquet files, (c) sub-second fraud alerts, (d) 5 PB archive. State the workload traits first.
- **G. MapReduce.** Sketch a Map/Reduce revenue count. Why does it not need Hadoop?